## short description

File `tahoe_sci_op3.pkl` contains the data from 3 datasets: tahoe, sciplex3 and op3.
It consists of 10 columns:
* `perturbagen` a name of perturbagen in the dataset specified in the column `dataset`. This name is the processed one, for querying you need to use `original_pert_name`
* `pubchem_cid` a fetched `pubchem_cid` from our `op3_v2` pipeline
* `smiles` smiles which were fetched based on `pubchem_cid`
* `dataset` is a name of dataset: tahoe, sciplex and op3.
* `cmap_name` is the name of the compound from a CLUE resource
* `symbol` is a name of drug in the vocab of LPM model
* `code` is index for the related embedding of LPM model
* `symbol_` is a processed `symbol` column
* `ECFP:2` is a fingerprint
* `LPM_emb` is LPM model embedding
* `original_pert_name` is a perturbagen original names before any processing. They are needed to query embeddings based on drug names.

File `dili.pkl` contains the data from dili dataset.
It includes the same columns represented in `DILI_from_web.csv` (the index colums is renamed to `pert_id`), and it was enriched with additional columns related to the mapping between compound names and embeddings:


* `pubchem_cid` a fetched `pubchem_cid` from our `op3_v2` pipeline
* `retrieved_smiles` smiles which were fetched based on `pubchem_cid`
* `cmap_name` is the name of the compound from a CLUE resource
* `symbol` is a name of drug in the vocab of LPM model
* `code` is index for the related embedding of LPM model
* `symbol_` is a processed `symbol` column
* `ECFP:2` is a fingerprint
* `LPM_emb` is LPM model embedding

**NB!** fingerprints were calculated based on both: existing `smiles` already represented in `DILI_from_web.csv` and `retrieved_smiles`, prioritizing existing `smiles` (if calculation for the compound based on `smiles` column is failed the calculation on `retrieved_smiles` was performed).

To read files you need to run:
```
import pandas as pd
pd.read_pickle("tahoe_sci_op3_updated.pkl")
```

## model embeddings

To get model embeddings from the saved model you need:

`import perturb_lib as plib`

`model = plib.load_trained_model('path2dir_containing_model_state/model.pt')`

`df_pert = model.vocab.perturb_vocab.to_pandas()`

`embeddings = model.perturb_embedding_layer.weight.numpy().astype(np.float64)`

## Supplementary 1.
Only for `tahoe_sci_op3.pkl`:

To reproduce and get `ECFP:2` fingerprints you need to run:

```
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from rdkit.DataStructs import ConvertToNumpyArray

def smiles_to_fingerprints(smiles_list, radius=1, fp_size=2000):
    """Convert a list of SMILES strings to Morgan (ECFP) fingerprint matrix.

    Args:
        smiles_list: Iterable of SMILES strings.
        radius: Morgan fingerprint radius (default 1, i.e. ECFP2).
        fp_size: Fingerprint bit vector size.

    Returns:
        np.ndarray of shape (len(smiles_list), fp_size), dtype uint8.
        Invalid SMILES yield a zero vector for that row.
    """
    gen = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=fp_size)
    fps = []
    for smile in smiles_list:
        mol = Chem.MolFromSmiles(smile)
        if mol is None:
            fps.append(None)
        else:
            fp = gen.GetFingerprint(mol)
            arr = np.zeros(fp_size, dtype=np.uint8)
            ConvertToNumpyArray(fp, arr)
            fps.append(arr)
    return fps

df['ECFP:2'] = smiles_to_fingerprints(df['smiles'])
```


Only for `dili.pkl`:

To reproduce and get `ECFP:2` fingerprints you need to run:

```
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from rdkit.DataStructs import ConvertToNumpyArray

def smiles_to_fingerprints(smiles_list, radius=1, fp_size=2000):
    """Convert a list of SMILES strings to Morgan (ECFP) fingerprint matrix.

    Args:
        smiles_list: Iterable of SMILES strings.
        radius: Morgan fingerprint radius (default 1, i.e. ECFP2).
        fp_size: Fingerprint bit vector size.

    Returns:
        np.ndarray of shape (len(smiles_list), fp_size), dtype uint8.
        Invalid SMILES yield a zero vector for that row.
    """
    gen = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=fp_size)
    fps = []
    for smiles in smiles_list:
        try:
            mol0 = Chem.MolFromSmiles(smiles[0])
        except:
            mol0 = None
        try:
            mol1 = Chem.MolFromSmiles(smiles[1])
        except:
            mol1 = None
        if mol0 is not None:
            fp = gen.GetFingerprint(mol0)
            arr = np.zeros(fp_size, dtype=np.uint8)
            ConvertToNumpyArray(fp, arr)
            fps.append(arr)
        elif mol1 is not None:
            fp = gen.GetFingerprint(mol1)
            arr = np.zeros(fp_size, dtype=np.uint8)
            ConvertToNumpyArray(fp, arr)
            fps.append(arr)
        else:
            fps.append(None)
    return fps

df['ECPF:2'] = smiles_to_fingerprints(df[['smiles', 'retrieved_smiles']].values)
```

## Supplementary 2. 
Only for `tahoe_sci_op3.pkl`:

**NB!** Some names in `perturbagen` column might be a bit different from the original names, as I took them not from the primaraly resource but after laminDB standardization and op3_v2 processing. Therefore, it is better to use `original_pert_name` for searching embeddings.


## Supplementary 3.

Only for `tahoe_sci_op3.pkl`:

**NB!** The following `perturbagen`: `tanespimycin` was mapped to a couple of LPM embeddings probably because they were linked to the same pubchem_cid. The cos similarity between these 2 embeddings is ~0.4, probably we can drop one of them to archieve 1-to-1 mapping for this compound, or maybe we can compute the average for this two embeddings and assign to `tanespimycin`.